# Code Generation with GraphRAG

## Introduction

In this notebook we show how to GraphRAG can be used to traverse through python library documentation in order to provide help to an LLM that is generating code.

We use a custom Strategy to traverse through the documents and specifically select nodes that contain code examples and/or descriptive text.

## Environment Setup

The following block will configure the environment from the Colab Secrets.
To run it, you should have the following Colab Secrets defined and accessible to this notebook:

- `OPENAI_API_KEY`: The OpenAI key.
- `ASTRA_DB_API_ENDPOINT`: The Astra DB API endpoint.
- `ASTRA_DB_APPLICATION_TOKEN`: The Astra DB Application token.
- `LANGCHAIN_API_KEY`: Optional. If defined, will enable LangSmith tracing.
- `ASTRA_DB_KEYSPACE`: Optional. If defined, will specify the Astra DB keyspace. If not defined, will use the default.

In [ ]:
# Install modules.
#
# On Apple hardware, "spacy[apple]" will improve performance.
%pip install \
    langchain-core \
    langchain-astradb \
    langchain-openai \
    langchain-graph-retriever \
    spacy \
    graph-rag-example-helpers

The last package -- `graph-rag-example-helpers` -- includes some helpers for setting up the environment and provides the documents we will use for this example.

In [ ]:
# Configure import paths.
import os
import sys

sys.path.append("../../")

# Initialize environment variables.
from graph_rag_example_helpers.env import Environment, initialize_environment

initialize_environment(Environment.ASTRAPY)

os.environ["LANGCHAIN_PROJECT"] = "code-generation"

## Part 1: Loading Data

First, we'll demonstrate how to load the example AstraPy documentation into `AstraDBVectorStore`. We will be creating a document for every module, class, attribute, and function in the package. 

Note that we will use the pydoc description field for the page_content field in the document. Because of this, we will have many documents that have no page content. We will store the rest of the item details in the document metadata.

#### Example doc with page content

<details markdown><summary>Click to expand</summary>

```yaml
id: astrapy.client.DataAPIClient

page_content: |
    A client for using the Data API. This is the main entry point and sits
    at the top of the conceptual "client -> database -> collection" hierarchy.

    A client is created first, optionally passing it a suitable Access Token.
    Starting from the client, then:
        - databases (Database and AsyncDatabase) are created for working with data
        - AstraDBAdmin objects can be created for admin-level work

metadata:
    name: DataAPIClient
    kind: class
    path: astrapy.client.DataAPIClient
    parameters: 
        token: |
            str | TokenProvider | None = None
            an Access Token to the database. Example: `"AstraCS:xyz..."`.
            This can be either a literal token string or a subclass of
            `astrapy.authentication.TokenProvider`.
        
        environment: |
            str | None = None
            a string representing the target Data API environment.
            It can be left unspecified for the default value of `Environment.PROD`;
            other values include `Environment.OTHER`, `Environment.DSE`.
        
        callers: |
            Sequence[CallerType] = []
            a list of caller identities, i.e. applications, or frameworks,
            on behalf of which Data API and DevOps API calls are performed.
            These end up in the request user-agent.
            Each caller identity is a ("caller_name", "caller_version") pair.

    example: |
        >>> from astrapy import DataAPIClient
        >>> my_client = DataAPIClient("AstraCS:...")
        >>> my_db0 = my_client.get_database(
        ...     "https://01234567-....apps.astra.datastax.com"
        ... )
        >>> my_coll = my_db0.create_collection("movies", dimension=2)
        >>> my_coll.insert_one({"title": "The Title", "$vector": [0.1, 0.3]})
        >>> my_db1 = my_client.get_database("01234567-...")
        >>> my_db2 = my_client.get_database("01234567-...", region="us-east1")
        >>> my_adm0 = my_client.get_admin()
        >>> my_adm1 = my_client.get_admin(token=more_powerful_token_override)
        >>> database_list = my_adm0.list_databases()

    references: 
        astrapy.client.DataAPIClient

    gathered_types: 
        astrapy.constants.CallerType
        astrapy.authentication.TokenProvider
```
</details>

This is the documentation for `astrapy.client.DataAPIClient`. The `page_content` field contains the description of the class, and the `metadata` field contains the rest of the details.

The `references` metadata field contains the list of related items used in the example code block. The `gathered_types` field contains the list of types from the parameters section. In GraphRAG, we can use these fields to link to other documents.

#### Example doc without page content

<details markdown><summary>Click to expand</summary>

```yaml
id: astrapy.admin.AstraDBAdmin.callers

page_content: ""

metadata:
  name: callers
  path: astrapy.admin.AstraDBAdmin.callers
  kind: attribute
```

</details>

This is the documentation for `astrapy.admin.AstraDBAdmin.callers`. The `page_content` field is empty, and the `metadata` field contains the details.

Despite having no page content, this document can still be useful.  We'll add a `parent` field to the metadata at vector store insertion time to link it to the parent document: `astrapy.admin.AstraDBAdmin`, and we can use this to traverse the graph.

### Create the AstraDBVectorStore
Next, we create the Vector Store we're going to load these documents into.
In our case, we use DataStax Astra DB with Open AI embeddings.

In [2]:
from langchain_astradb import AstraDBVectorStore
from langchain_openai import OpenAIEmbeddings

store = AstraDBVectorStore(
    embedding=OpenAIEmbeddings(),
    collection_name="code_generation"
)

### Loading Data into the Store
Next, we perform the actual loading. There are only about 1000 documents, so this should be quick.

In [3]:
from graph_rag_example_helpers.datasets.astrapy import fetch_documents

store.add_documents(fetch_documents())

HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/datastax/graph-rag/refs/heads/main/data/astrapy.jsonl